# Drift correction tutorial - 0°/90° HAADF pair (real EMD)

A STEM scan drifts during acquisition, so a single image is sheared by sample +
stage motion. Collecting the **same area twice at orthogonal scan rotations (0° and
90°)** lets us solve that per-scanline drift: each scan smears drift along its own
fast-scan axis, so the orthogonal partner constrains what the first cannot. This
notebook walks the full pipeline on a real Velox EMD pair and explains every parameter.

All figures here come from `DriftCorrection`'s **own plotting** (`show_merged=...`,
`plot_merged_images`, `plot_warped_images`) for static PNG views, plus `Show2D` for the
interactive comparison.

**Data:** `20260515_dram_drift_dasol`, 15.0 Mx, 6.74 nm FOV, 2048×2048 - the KENTECH demo pair (`0041`=0°, `0042`=90°).

**Pipeline:** `read_emd_pair` → `preprocess` → `align_affine` → diagnostics → corrected merge.

In [ ]:
# Install for this tutorial:
#  - quantem: from the drift branch of the bobleesj fork (read_emd_pair + DriftCorrection
#    live here; not yet in a released quantem).
#  - quantem.widget (Show2D): latest from TestPyPI.
# Restart the kernel after this cell on a fresh environment.
%pip install -q -U "quantem @ git+https://github.com/bobleesj/quantem@drift-june-2026"
%pip install -q -U -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ quantem-widget

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import quantem as em
from quantem.widget import Show2D

## 1. Load the 0°/90° pair

`em.imaging.read_emd_pair(path_0, path_90)` reads both Velox EMD files and returns a
bundle. The scan rotation is read **straight from each file's `Scan.ScanRotation`
metadata** - you never type the angle. The returned `scan_direction_degrees` (here
`(0.0, 90.0)`) is exactly the form `DriftCorrection` needs to orient each scan.

- `data` - the two frames as `Dataset2d` (each carries its nm pixel size)
- `scan_direction_degrees` - per-frame scan rotation in degrees, auto-parsed
- `pixel_size_nm`, `shape` - calibration straight from the microscope

In [ ]:
DATA_DIR = '/home/owner/data/dasol/20260515_dram_drift_dasol'
# KENTECH demo pair: 15.0 Mx, 6.74 nm FOV, 2048x2048 (0041 = 0 deg, 0042 = 90 deg).
F0  = '0041-20260515_1130_15.0_Mx_6.74_nm_Nano_HAADF.emd'
F90 = '0042-20260515_1130_15.0_Mx_6.74_nm_Nano_HAADF.emd'

# read_emd_pair prints shape / pixel size / scan angles itself (verbose=True default).
pair = em.imaging.read_emd_pair(f'{DATA_DIR}/{F0}', f'{DATA_DIR}/{F90}')
data_0, data_1 = pair['data']

## 2. `preprocess` - build the scanline model

`DriftCorrection.from_images(im0, im90, scan_direction_degrees=...)` creates the solver;
`preprocess` lays both scans onto a shared padded canvas and initializes the Bezier
knots that map each scanline into it. Nothing can be aligned before this. This is pure
setup - the meaningful merge appears after `align_affine`, so we skip its plots here.

**Parameters**
- `pad_fraction=0.5` - canvas padding per side (50%). Bigger = more room for drift, more memory.
- `pad_value='median'` - fill outside each scan's footprint (keeps background flat).
- `kde_sigma=0.5` - Gaussian smoothing (px) after the bilinear scatter; suppresses scatter noise.
- `number_knots=1` - knots per scanline. `1` = **linear** drift per line (recommended).

In [ ]:
drift = em.imaging.DriftCorrection.from_images(
    data_0, data_1, scan_direction_degrees=pair['scan_direction_degrees'],
)
drift.preprocess(pad_fraction=0.5, pad_value='median', kde_sigma=0.5, number_knots=1)

## 3. `align_affine` - grid-search the per-scanline drift

Builds a grid of candidate linear-drift rates, warps both scans for each, keeps the
lowest cross-correlation cost; `refine` subdivides the winner for sub-step accuracy.
`show_merged=True` makes the class plot the corrected merge as soon as it finishes.

**Parameters**
- `step=0.02` - grid resolution in **px per scan line**.
- `num_tests=21` - rates tested per axis (odd, centered on zero). Grid spans
  `± step × (num_tests-1)/2` = **±0.20 px/line** here.
- `refine=True` - second pass at `step/(num_tests-1)` around the winner.
- `max_image_shift=512` - reject cross-correlation peaks beyond this radius (px). **Must be
  large enough for the real shift** - too small silently caps the result.

> **Grid-width rule:** `step × (num_tests-1)/2` must be ≥ the true max drift, or the search
> lands on the boundary and *undershoots*. ±0.10 was too narrow for this pair; ±0.20 recovers it.

**Readouts**
- `drift_rate` - solved (row, col) drift in px/line.
- `affine_confidence_margin` - % cost gap to the runner-up candidate. Low on periodic atomic
  lattices (many near-equal candidates) - expected here, not a failure.

In [ ]:
# align_affine prints the solved drift rate, total shift, and confidence itself.
drift.align_affine(step=0.02, num_tests=21, refine=True, max_image_shift=512, show_merged=True)

## 4. Diagnostic - RGB overlay

`align_affine(show_merged=True)` already showed the plain corrected merge, so the only
extra view worth one cell is the RGB overlay: the two corrected scans go in separate
color channels - **agreement is gray, any residual offset shows as colored fringes**.

In [ ]:
drift.plot_merged_images(axsize=(8, 8), rgb=True)

## 5. Corrected merge - interactive

`generate_corrected` warps both scans with the solved drift and averages them into a
single drift-free image. `Show2D` puts raw 0°, raw 90°, and the corrected merge side by
side, **zoomed 3× with a calibrated scale bar** (the nm/px pixel size comes straight from
the EMD), with shared pan / zoom / contrast in the browser.

In [ ]:
image_corr = drift.generate_corrected(show_merged=False)
Show2D(
    [data_0.array, data_1.array, image_corr.array],
    labels=['Raw 0°', 'Raw 90°', 'Drift-corrected merge'],
    sampling=pair['pixel_size_nm'],   # nm/px from the EMD, so the scale bar is correct
    units='nm',
    ncols=3,                          # 3 panels across
    size=400,                         # 400 px per panel
    zoom=3.0,                         # 3x magnification built in
)

## 6. Save the corrected merge

In [ ]:
from pathlib import Path
save_dir = Path('outputs'); save_dir.mkdir(exist_ok=True)
out = save_dir / 'drift_dasol_15Mx_0041_0042_merged.npy'
np.save(out, image_corr.array)
print(f'saved {out}  shape={image_corr.array.shape}  dtype={image_corr.array.dtype}')